In [ ]:
from pypdf import PdfReader
import os
dir = '/media/system/ZERBUIS_EXT_STOR/temp/exp/transformer/papers'
paths = [os.path.join(dir, f) for f in os.listdir(dir)]

text = ""
for path in paths:
    reader = PdfReader(path)
    for page in reader.pages:
        text += page.extract_text()

print(text)

print(len(text))
with open("data.txt", 'w') as file:
    file.write(text)

In [ ]:
print(len(text))
with open("data.txt", 'w') as file:
    file.write(text)

bpe

In [1]:
from icecream import ic
import copy

In [2]:
with open('data.txt', 'r') as f:
    text = f.read()
text

'OCR-free Document Understanding Transformer\nGeewook Kim1∗, Teakgyu Hong4†, Moonbin Yim2†, Jeongyeon Nam1,\nJinyoung Park5†, Jinyeong Yim6†, Wonseok Hwang7†, Sangdoo Yun3,\nDongyoon Han3, and Seunghyun Park1\n1NAVER CLOVA 2NAVER Search 3NAVER AI Lab\n4Upstage 5Tmax 6Google 7LBox\nAbstract. Understanding document images (e.g., invoices) is a core but\nchallenging task since it requires complex functions such as reading text\nand a holistic understanding of the document. Current Visual Document\nUnderstanding (VDU) methods outsource the task of reading text to off-\nthe-shelf Optical Character Recognition (OCR) engines and focus on the\nunderstanding task with the OCR outputs. Although such OCR-based\napproaches have shown promising performance, they suffer from 1) high\ncomputational costs for using OCR; 2) inflexibility of OCR models on\nlanguages or types of documents; 3) OCR error propagation to the sub-\nsequent process. To address these issues, in this paper, we introduce a\nnovel

In [3]:
tokens = text.encode('utf-8')
tokens = list(map(int, tokens))
len(tokens), len(text)

(1003579, 986964)

In [4]:
def get_stats(text):
    count = {}
    for pair in zip(text, text[1:]):
        count[pair] = count.get(pair, 0) + 1
    return count 

tokens_stats = get_stats(copy.deepcopy(tokens))

In [5]:
stats = sorted(tokens_stats.items(), key=lambda x: x[1], reverse=True)[:10]

1. find the pairs with maximum occurances
2. choose a hyperparameter K and keep combining the pairs to new token

In [42]:
sorted(tokens_stats.items(), key=lambda x: x[1], reverse=True)[:10]

[((44, 32), 2274),
 ((105, 110), 1398),
 ((101, 32), 1198),
 ((32, 97), 1089),
 ((46, 44), 994),
 ((110, 103), 928),
 ((97, 116), 897),
 ((97, 110), 888),
 ((111, 110), 859),
 ((32, 116), 798)]

In [6]:
combined_bpe = {} # pair, new_id
count = 256 # maximum value of utf-8

def merge(ids, pair, idx):
  # in the list of ints (ids), replace all consecutive occurences of pair with the new token idx
  newids = []
  i = 0
  while i < len(ids):
    # if we are not at the very last position AND the pair matches, replace it
    if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
      newids.append(idx)
      i += 2
    else:
      newids.append(ids[i])
      i += 1
  return newids


new_tokens = merge(tokens, stats[0][0], count)

In [7]:
ic(len(new_tokens))
ic(stats[0][-1])
ic(len(tokens) - len(new_tokens))

ic| len(new_tokens): 984268
ic| stats[0][-1]: 19311
ic| len(tokens) - len(new_tokens): 19311


19311

In [9]:
total_merges = 2500
new_ids = tokens.copy()
count = 256
merges = {} # idx, pair
while total_merges > 0:
    tokens_stats = get_stats(new_ids)
    max_pair = sorted(tokens_stats.items(), key=lambda x: x[1], reverse=True)[0]
    print(f'Merging {max_pair[0]} with frequency {max_pair[1]}, to {count}')
    merges[count] = max_pair[0]
    new_ids = merge(new_ids, max_pair[0], count)
    count += 1
    total_merges -= 1

print(f'Initial tokens lenght: {len(tokens)}, after merging {len(new_ids)}')

Merging (101, 32) with frequency 19311, to 256
Merging (44, 32) with frequency 15411, to 257
Merging (105, 110) with frequency 14975, to 258
Merging (111, 110) with frequency 12270, to 259
Merging (116, 104) with frequency 11660, to 260
Merging (115, 32) with frequency 11393, to 261
Merging (97, 110) with frequency 10632, to 262
Merging (101, 114) with frequency 9829, to 263
Merging (116, 105) with frequency 8956, to 264
Merging (100, 32) with frequency 8899, to 265
Merging (101, 110) with frequency 8879, to 266
Merging (116, 32) with frequency 8074, to 267
Merging (111, 114) with frequency 6920, to 268
Merging (46, 32) with frequency 6875, to 269
Merging (97, 108) with frequency 6542, to 270
Merging (258, 103) with frequency 6299, to 271
Merging (114, 101) with frequency 6287, to 272
Merging (97, 114) with frequency 5925, to 273
Merging (260, 256) with frequency 5640, to 274
Merging (264, 259) with frequency 5573, to 275
Merging (271, 32) with frequency 4352, to 276
Merging (97, 116) 

In [66]:
bytes([50]).decode('utf-8') # sample decoding

'2'

In [20]:
len(set(new_ids))

2682

In [11]:
def expand(token, merge_to_pair):
    if token < 256:
        return [token]

    left, right = merge_to_pair[token]
    return expand(left, merge_to_pair) + expand(right, merge_to_pair)

def decode(tokens, merge_to_pair):
    bytes_list = []

    for token in tokens:
        bytes_list.extend(expand(token, merge_to_pair))

    return bytes(bytes_list).decode("utf-8")

In [19]:
bytes(expand(2169, merges)).decode("utf-8")

're-\n'